# CatBoost fold pipeline — template, not yet run

**Author:** SF — **Branch:** `foldingstrategiesv2` — **Scope:** a **template**. `CONFIG
["run_training"]` is `False`; every cell that would actually fit a model is gated behind
it and prints a "skipped" message instead of running. This notebook is safe to "Run All"
today — it builds and inspects the splits, and stops short of training anything.

## Why CatBoost specifically, not LightGBM/XGBoost

The three baselines already tested (`LogisticRegression`, `RandomForestClassifier`,
`HistGradientBoostingClassifier`, in `sf_eda_v2.ipynb`/`sf_logo_fold_strategy.ipynb`) all
collapse similarly under Leave-One-Use-Case-Out — a different tree-boosting library isn't
expected to close that domain-shift gap; this is not a fix for it. What CatBoost adds is
narrower and more specific than "a more advanced tree ensemble" in general:

- **`RandomForestClassifier` and `HistGradientBoostingClassifier` both hit train AUC =
  1.000 in every LOGO rotation** (`sf_logo_fold_strategy.ipynb` §3.2, `sf_logo_fold_pipeline.ipynb`
  §4) — severe in-sample overfitting. Ordinary gradient boosting (including
  `HistGradientBoostingClassifier`, which sklearn's own docs describe as inspired by
  LightGBM — so LightGBM is not a meaningfully different fourth model here) computes each
  tree's target statistics from the *entire* training set, a well-documented source of
  target leakage during training on smaller datasets. **CatBoost's ordered boosting
  computes those statistics on a permuted prefix of the data instead**, specifically to
  suppress that leakage — a mechanism aimed at the exact failure mode already measured on
  this data, not a different vendor's implementation of the same one.
- **Native categorical handling via leakage-safe target statistics**, not one-hot or a
  bare `category`-dtype cast — a better match than LightGBM's approach once the
  feature-engineering track adds categorical columns (e.g. `domain_industry`-style
  fields).
- Same offline/CPU-only, monotonic-constraints, mature-SHAP story as any modern gradient
  booster — see this session's model-recommendation writeup for the full comparison
  against the prompted-LLM pick.

**Still true regardless of library:** this is a production/interpretability upgrade to
the tree-ensemble arm this project already trusts, evaluated with the same LOGO discipline
to see whether it helps at all — not assumed to fix generalization to a new use case on
its own.

## What this notebook does

Combines both fold strategies already built in `notebooks/pipelines/` — the pooled/
generalized split and the Leave-One-Use-Case-Out rotation — against a `CatBoostClassifier`
instead of `LogisticRegression`, reusing `scripts/fold_pipeline_utils.py`'s
`build_tree_pipeline`/`build_tree_preprocessor` (no scaling/imputation, categorical
columns cast to pandas `category` dtype, `boosting_type="Ordered"` pinned explicitly
rather than left to CatBoost's own size-based heuristic). Same split logic, same leakage
checks, same metrics (`classification_metrics`/`ranking_metrics`/`bootstrap_auc_ci`) as the
`LogisticRegression` pipelines — only the model/preprocessing step differs.

**Not executed in this session — `pip install catboost` first** (not yet in
`requirements.txt`; uncomment it once this notebook is actually run).


In [ ]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("../../scripts").resolve()
sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

from fold_pipeline_utils import (
    bootstrap_auc_ci,
    build_tree_pipeline,
    classification_metrics,
    prepare_dataset,
    ranking_metrics,
    validate_schema,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## CONFIG

Same shape as `sf_generalized_fold_pipeline.ipynb`/`sf_logo_fold_pipeline.ipynb`, plus
`run_training` — the template switch. Leave it `False` until you're ready to actually fit
`CatBoostClassifier` and look at real numbers.


In [ ]:
CONFIG = {
    "data_path": "../../data/processed/papers_combined.parquet",
    # target
    "target_col": "triage_label",
    "positive_label": "positive",
    "negative_labels": ["negative"],
    # columns used by the split/leakage logic, not as model features
    "use_case_col": "use_case_key",
    "group_col": "first_author",
    "group_source_col": "authors",
    "title_col": "title",
    # model features - extend these as feature engineering lands new columns.
    # CatBoost needs no scaling and handles missing numeric values natively; any column
    # named here as categorical is cast to pandas `category` dtype AND passed to
    # CatBoostClassifier's own `cat_features` (see build_tree_pipeline), so it's encoded
    # via CatBoost's leakage-safe ordered target statistics, not one-hot.
    "embedding_col": "embedding",
    "expand_embedding": True,
    "numeric_feature_cols": ["citation_velocity"],
    "categorical_feature_cols": [],   # e.g. ["domain_industry"] once/if used as a feature
    # split sizes
    "n_splits_outer": 5,
    "n_splits_inner": 5,
    "random_state": 0,
    # TEMPLATE SWITCH - leave False until ready to actually train/evaluate CatBoost.
    # Every cell that fits a model checks this and prints a skip message instead of
    # running when it's False - safe to "Run All" today.
    "run_training": False,
}


## 0. Load data, and a stand-in for the still-in-progress feature engineering

**Temporary — delete once the real engineered dataset lands**, identical stopgap to
`notebooks/pipelines/*.ipynb` §0.


In [ ]:
df_raw = pd.read_parquet(CONFIG["data_path"])
print(f"Loaded {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns from {CONFIG['data_path']}")

# --- TEMPORARY stand-in, remove once the real engineered dataset provides this column ---
df_raw["exported_year"] = pd.to_datetime(df_raw["exported_at"]).dt.year
df_raw["paper_age"] = (df_raw["exported_year"] - df_raw["year"]).clip(lower=0)
df_raw["citation_velocity"] = df_raw["citation_count"] / (df_raw["paper_age"].astype("float") + 1)
# --- end temporary stand-in ---


## 1. Schema check + row-wise prep — safe on the WHOLE dataset, before any split

Same two functions as both `pipelines/` notebooks §1 — `validate_schema` fails loudly on a
CONFIG/dataset mismatch; `prepare_dataset` only does per-row, parameter-free work.


In [ ]:
validate_schema(df_raw, CONFIG)
df = prepare_dataset(df_raw, CONFIG)
USE_CASES = sorted(df[CONFIG["use_case_col"]].unique().tolist())
FEATURE_COLS = CONFIG["_embedding_feature_cols"] + CONFIG["numeric_feature_cols"] + CONFIG["categorical_feature_cols"]
print(f"{len(df):,} labelled rows (positive/negative only) across {len(USE_CASES)} use cases")
print(f"{len(FEATURE_COLS)} feature columns going into CatBoost (no scaling/imputation needed)")


## 2. Duplicate-row leakage check

Same check as both `pipelines/` notebooks §2 — model-agnostic, so it's identical here.


In [ ]:
title_norm = df[CONFIG["title_col"]].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
n_dup_titles = title_norm.duplicated(keep=False).sum()
print(f"{n_dup_titles} rows share a title with at least one other row in the pooled dataset "
      f"(out of {len(df):,})")


## 3. Where the split needs to happen, and why

Same principle as the `LogisticRegression` pipelines, with one simplification: CatBoost
needs no scaling/imputation, so there's less that *could* leak on the transformation side
today. The rule still applies to anything fitted in the future — if the feature-engineering
track ships a percentile rank, a target encoding, or any other fitted statistic, it still
must be fit per-fold, inside the split, exactly like `StandardScaler` was for the
`LogisticRegression` pipelines. Nothing above this line fits anything; nothing below it may
fit using rows outside the fold/rotation currently being trained on.

### Part A — generalized (pooled) split


In [ ]:
strat_key_outer = df[CONFIG["use_case_col"]].astype(str) + "__" + df["y"].astype(str)
outer_cv = StratifiedGroupKFold(
    n_splits=CONFIG["n_splits_outer"], shuffle=True, random_state=CONFIG["random_state"]
)
outer_splits = list(outer_cv.split(df, strat_key_outer, groups=df[CONFIG["group_col"]]))
train_pool_idx, final_holdout_idx = outer_splits[0]

train_pool_a = df.iloc[train_pool_idx].reset_index(drop=True)
final_holdout_a = df.iloc[final_holdout_idx].reset_index(drop=True)
print(f"Part A: train_pool {len(train_pool_a):,} rows | final_holdout {len(final_holdout_a):,} rows")

strat_key_inner_a = train_pool_a[CONFIG["use_case_col"]].astype(str) + "__" + train_pool_a["y"].astype(str)
inner_cv_a = StratifiedGroupKFold(
    n_splits=CONFIG["n_splits_inner"], shuffle=True, random_state=CONFIG["random_state"]
)
inner_splits_a = list(inner_cv_a.split(train_pool_a, strat_key_inner_a, groups=train_pool_a[CONFIG["group_col"]]))
print(f"Part A: {len(inner_splits_a)} inner validation folds built within train_pool")


### Part B — Leave-One-Use-Case-Out rotation

In [ ]:
print(f"Part B: {len(USE_CASES)} rotations, one per use case: {USE_CASES}")


## 4. The model, and why fitting it is gated

`build_tree_pipeline(CONFIG)` returns a `Pipeline([("preprocess", <categorical caster>),
("clf", CatBoostClassifier(...))])` — same calling convention as `build_pipeline` in the
`LogisticRegression` notebooks (`.fit(X_train, y_train)` fits everything in one call).
`CatBoostClassifier` is constructed with `auto_class_weights="Balanced"` (matching every
other classifier in this repo's `sf_*` lineage), `boosting_type="Ordered"` (pinned
explicitly — see this notebook's intro for why that specific mechanism was the reason to
pick CatBoost at all), and `cat_features` read straight from
`CONFIG["categorical_feature_cols"]`. Requires `catboost` to be installed - **not done in
this session**, since `CONFIG["run_training"]` is `False` and nothing below actually calls
`build_tree_pipeline` yet outside the gated block.


In [ ]:
if CONFIG["run_training"]:
    fold_pipeline = build_tree_pipeline(CONFIG)
    print(fold_pipeline)
else:
    print("Skipped: CONFIG['run_training'] is False. This is a template - flip it to True, "
          "`pip install catboost`, and re-run once ready to actually train/evaluate.")


## 5. Part A — train / validation / holdout (gated)

Structurally identical to `sf_generalized_fold_pipeline.ipynb` §4-§5: inner CV over
`inner_splits_a` for validation, one `build_tree_pipeline(CONFIG)` fit on the whole of
`train_pool_a` for the train stage, the same fitted pipeline scored on `final_holdout_a`
once for holdout. All gated — see §4.


In [ ]:
if CONFIG["run_training"]:
    val_rows_a = []
    for tr_idx, te_idx in inner_splits_a:
        fold_pipeline = build_tree_pipeline(CONFIG)
        fold_train, fold_test = train_pool_a.iloc[tr_idx], train_pool_a.iloc[te_idx]
        fold_pipeline.fit(fold_train[FEATURE_COLS], fold_train["y"])
        pos_col = list(fold_pipeline.classes_).index(1)
        proba = fold_pipeline.predict_proba(fold_test[FEATURE_COLS])[:, pos_col]
        pred = fold_pipeline.predict(fold_test[FEATURE_COLS])
        val_rows_a.append(classification_metrics(fold_test["y"].to_numpy(), proba, pred))
    val_df_a = pd.DataFrame(val_rows_a)

    final_pipeline_a = build_tree_pipeline(CONFIG)
    final_pipeline_a.fit(train_pool_a[FEATURE_COLS], train_pool_a["y"])
    pos_col_a = list(final_pipeline_a.classes_).index(1)

    train_proba_a = final_pipeline_a.predict_proba(train_pool_a[FEATURE_COLS])[:, pos_col_a]
    train_pred_a = final_pipeline_a.predict(train_pool_a[FEATURE_COLS])
    train_metrics_a = classification_metrics(train_pool_a["y"].to_numpy(), train_proba_a, train_pred_a)

    holdout_proba_a = final_pipeline_a.predict_proba(final_holdout_a[FEATURE_COLS])[:, pos_col_a]
    holdout_pred_a = final_pipeline_a.predict(final_holdout_a[FEATURE_COLS])
    holdout_metrics_a = classification_metrics(final_holdout_a["y"].to_numpy(), holdout_proba_a, holdout_pred_a)
    holdout_metrics_a.update(ranking_metrics(final_holdout_a["y"].to_numpy(), holdout_proba_a))
    ci_lo_a, ci_hi_a = bootstrap_auc_ci(final_holdout_a["y"].to_numpy(), holdout_proba_a)
    holdout_metrics_a["auc_ci_low"], holdout_metrics_a["auc_ci_high"] = ci_lo_a, ci_hi_a

    stage_table_a = pd.DataFrame({
        "train": {k: train_metrics_a[k] for k in ["auc", "ap", "recall", "f2"]},
        "validation_mean": {k: val_df_a[k].mean() for k in ["auc", "ap", "recall", "f2"]},
        "validation_std": {k: val_df_a[k].std() for k in ["auc", "ap", "recall", "f2"]},
        "holdout": {k: holdout_metrics_a[k] for k in ["auc", "ap", "recall", "f2"]},
    }).T.round(3)
    print("Part A: train -> validation -> holdout")
    display(stage_table_a)
else:
    print("Skipped: CONFIG['run_training'] is False.")


## 6. Part B — LOGO rotation (gated)

Structurally identical to `sf_logo_fold_pipeline.ipynb` §4: for each use case, hold it out
entirely, run inner CV on the rest for validation, fit one `build_tree_pipeline(CONFIG)`
on the whole `dev_pool` for train + holdout. All gated — see §4.


In [ ]:
if CONFIG["run_training"]:
    logo_rows = []
    for holdout_uc in USE_CASES:
        dev_pool = df[df[CONFIG["use_case_col"]] != holdout_uc].reset_index(drop=True)
        holdout = df[df[CONFIG["use_case_col"]] == holdout_uc].reset_index(drop=True)

        strat_key = dev_pool[CONFIG["use_case_col"]].astype(str) + "__" + dev_pool["y"].astype(str)
        inner_cv = StratifiedGroupKFold(
            n_splits=CONFIG["n_splits_inner"], shuffle=True, random_state=CONFIG["random_state"]
        )
        inner_splits = list(inner_cv.split(dev_pool, strat_key, groups=dev_pool[CONFIG["group_col"]]))

        val_rows = []
        for tr_idx, te_idx in inner_splits:
            fold_pipeline = build_tree_pipeline(CONFIG)
            fold_train, fold_test = dev_pool.iloc[tr_idx], dev_pool.iloc[te_idx]
            fold_pipeline.fit(fold_train[FEATURE_COLS], fold_train["y"])
            pos_col_inner = list(fold_pipeline.classes_).index(1)
            proba = fold_pipeline.predict_proba(fold_test[FEATURE_COLS])[:, pos_col_inner]
            pred = fold_pipeline.predict(fold_test[FEATURE_COLS])
            val_rows.append(classification_metrics(fold_test["y"].to_numpy(), proba, pred))
        val_df_rotation = pd.DataFrame(val_rows)

        final_pipeline = build_tree_pipeline(CONFIG)
        final_pipeline.fit(dev_pool[FEATURE_COLS], dev_pool["y"])
        pos_col = list(final_pipeline.classes_).index(1)

        train_proba = final_pipeline.predict_proba(dev_pool[FEATURE_COLS])[:, pos_col]
        train_pred = final_pipeline.predict(dev_pool[FEATURE_COLS])
        train_metrics = classification_metrics(dev_pool["y"].to_numpy(), train_proba, train_pred)

        holdout_proba = final_pipeline.predict_proba(holdout[FEATURE_COLS])[:, pos_col]
        holdout_pred = final_pipeline.predict(holdout[FEATURE_COLS])
        holdout_metrics = classification_metrics(holdout["y"].to_numpy(), holdout_proba, holdout_pred)
        rm = ranking_metrics(holdout["y"].to_numpy(), holdout_proba)
        ci_lo, ci_hi = bootstrap_auc_ci(holdout["y"].to_numpy(), holdout_proba)

        logo_rows.append({
            "holdout_use_case": holdout_uc,
            "train_auc": round(train_metrics["auc"], 3),
            "validation_auc": round(float(val_df_rotation["auc"].mean()), 3),
            "holdout_auc": round(holdout_metrics["auc"], 3),
            "holdout_recall": round(holdout_metrics["recall"], 3),
            "holdout_f2": round(holdout_metrics["f2"], 3),
            **{k: round(v, 3) for k, v in rm.items()},
            "holdout_auc_ci_low": round(ci_lo, 3),
            "holdout_auc_ci_high": round(ci_hi, 3),
        })

    logo_results = pd.DataFrame(logo_rows)
    print("Part B: LOGO rotation summary")
    display(logo_results)
    display(logo_results[["train_auc", "validation_auc", "holdout_auc"]].agg(["mean", "std"]).round(3))
else:
    print("Skipped: CONFIG['run_training'] is False.")


## 7. Checklist — before flipping `run_training` to `True`

1. `pip install catboost` (add it to `requirements.txt` at the same time, with a comment
   noting it's the advanced-tree-ensemble candidate, same convention as this file's other
   dependency comments).
2. Everything from the `pipelines/` notebooks' own dataset-swap checklist still applies
   here unchanged: point `CONFIG["data_path"]` at the real engineered dataset, delete §0's
   temporary stand-in, extend `numeric_feature_cols`/`categorical_feature_cols`, re-run
   `validate_schema` first.
3. Consider `CatBoostClassifier` knobs this dataset's scale (a few hundred rows per use
   case) makes relevant before trusting a first run: `l2_leaf_reg` raised from the
   default and/or `depth` lowered (fight overfitting at this row count —
   `RandomForestClassifier`/`HistGradientBoostingClassifier` both hit train AUC 1.000 on
   this exact data), and `monotone_constraints` if a domain prior on any feature's
   direction is worth encoding.
4. Confirm `boosting_type="Ordered"` actually took effect (`model.get_params()` after a
   fit) rather than assuming it — this is the specific mechanism this notebook exists to
   test, not an incidental default.
5. Only then set `CONFIG["run_training"] = True` and re-run §4-§6.
6. **Expect the LOGO number (§6) to still collapse similarly to the existing baselines' 0.50–0.59
   range** — that's the honest prior from this project's own evidence trail, not a reason
   to skip running this. Ordered boosting might narrow the *overfitting* gap (train →
   validation) more than plain gradient boosting did; nothing about it has been shown to
   touch the *domain-shift* gap (validation → holdout), which every model tried so far
   shares.
